In [4]:
import os, json, argparse, math, random
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# import bitsandbytes as bnb
from sklearn.model_selection import train_test_split

In [5]:
def to_single_prompt(ex):
    # Accepts either instruction-style or raw OriginalComment/segmentText
    if "instruction" in ex and "input" in ex and "output" in ex:
        instr, inp, out = ex["instruction"], ex["input"], ex["output"]
    else:
        instr = "Segment the following feedback into distinct, semantically meaningful parts. Each segment should capture a single, complete idea related to one of the predefined themes (Advocacy, Bench Strength, Client Service, Commercial Awareness, Cross-border, Delivery to Budget, Diversity & Inclusion, Market Trends, or Sophistication). If a segment doesn’t clearly match any theme, label it as 'Undefined.' Maintain logical flow and coherence between segments."
        inp = ex.get("OriginalComment", "")
        out = ex.get("segmentText", "")
    return {"text": f"Instruction: {instr}\nInput: {inp}\nOutput: {out}"}

In [6]:

ds_all = load_dataset("json", data_files={"train": "data.json"})["train"]

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /Users/sathishkumarchandran/.cache/huggingface/datasets/json/default-72e283e04315aa01/0.0.0/e347ab1c932092252e717ff3f949105a4dd28b27e842dd53157d2f72e276c2e4. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
ds_all

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 10000
})

In [9]:
# Simple split (stratification not needed for instruction tuning)
idx = list(range(len(ds_all)))
random.seed(42); random.shuffle(idx)
split = int(len(idx) * 0.9)
train_idx, val_idx = idx[:split], idx[split:]
ds = DatasetDict({
    "train": ds_all.select(train_idx),
    "validation": ds_all.select(val_idx)
})


In [10]:
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 9000
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1000
    })
})

In [11]:
ds = ds.map(to_single_prompt, remove_columns=ds["train"].column_names)

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [12]:
ds

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 9000
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1000
    })
})

In [16]:
model_id="HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

In [18]:

def tok_batch(batch):
    return tok(batch["text"], max_length=1024, truncation=True)

ds_tok = ds.map(tok_batch, batched=True, remove_columns=["text"])

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [19]:
ds_tok

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 9000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1000
    })
})